<a href="https://colab.research.google.com/github/writetomangamsudheer-stack/ai-mentor-portfolio/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search

In [2]:
import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [3]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""

    return DuckDuckGoSearchRun().run(query)

/tmp/ipykernel_5400/2177967920.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [6]:
!pip install -q -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.6 MB/s eta 0:00:00


In [4]:
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

February 25, 2026 - TCS will conduct the hiring round using the TCS iON platform rather than the TCS Community platform, which TCS has utilized in the past. Yes, students from 2024 can apply in this drive. January 20, 2026 - Tata Consultancy Services (TCS), India’s most trusted IT services company, has launched its TCS Hiring 2026 drive, creating a massive wave of job opportunities for freshers an


In [5]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

agent = create_react_agent(
    llm,
    tools=[web_search]
)

print("Agent created.")

Agent created.


/tmp/ipykernel_5400/1216206272.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [6]:
result = agent.invoke({
    'messages': [
        ('user', "What is TCS's 2026 hiring quota?")
    ]
})

In [7]:
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')

    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')

    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': '64892386-728a-44a4-b514-3e752bd6b704', 'type': 'tool_call'}]

[2] ToolMessage
    Content: Discover the latest TCS hiring 2026 updates, including TCS recruitment 2026, upcoming TCS jobs 2026, and all new TCS vacancies 2026 across India. Find complete details on TCS freshers hiring, eligibility, roles, and how to apply for top IT openings. Apr 8, 2026 · Tata Consultancy Services (TCS) (BSE

[3] AIMessage
    Content: [{'type': 'text', 'text': 'TCS has made 25,000 job offers to fresh graduates for FY2026-27. The company aims to onboard around 40,000 freshers annually.', 'extras': {'signature': 'CvACAQw51seEAk6wC9brkuWZFEtNSKs3KYZuD5K/0eMc5M/yc6PS72ekhZQZPlzCcvByrJIj+WWmnXu8LXz9BvIohkYiMb2y5FBHmL9miuu1QEyA8Iou4bQL


In [8]:
result = agent.invoke({
    'messages': [
        ('user',
         'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')
    ]
})

In [9]:
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')

    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    

[2] ToolMessage
    March 20, 2026 - # ISPs sometimes hijack NXDOMAIN to return their own "helpful" search page # Symptom: NXDOMAIN query returns an IP instead of NXDOMAIN status # Test for NXDOMAIN hijacking: dig random-nonexistent-12345.invalid # Should return: NXDOMAIN # If returns IP: your ISP is hijacking NXDOMAIN

[3] AIMessage
    [{'type': 'text', 'text': 'The URL you provided, "https://this-domain-does-not-exist-12345.example.com/jd", appears to be for a domain that does not exist. The search results discuss "NXDOMAIN" errors, which indicate that a domain name cannot be found. This means there\'s no content to display at th


In [13]:
import requests
from bs4 import BeautifulSoup
from langchain_core.tools import tool

@tool
def jd_fetcher(url: str) -> str:
    """Fetch a job description from a URL and return clean plain text.
    Use when the user provides a job posting URL and you need the JD content.
    Returns first 4000 characters of the cleaned page text."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:4000]
    except Exception as e:
        return f'ERROR: failed to fetch URL — {e}'

In [14]:
@tool
def skills_gap(student_skills: str, must_have_skills: str) -> str:
    """Compare a student's skills (comma-separated) to a job's must-have skills (comma-separated).
    Returns missing skills, comma-separated, or 'none' if student has all.
    Use when the user provides a student profile and a JD's required skills."""
    a = set(s.strip().lower() for s in student_skills.split(',') if s.strip())
    b = set(s.strip().lower() for s in must_have_skills.split(',') if s.strip())
    missing = sorted(b - a)
    return ', '.join(missing) if missing else 'none'

# Test
print(skills_gap.invoke({
    'student_skills': 'Python, Java, SQL',
    'must_have_skills': 'Python, Java, SQL, Spring Boot, AWS',
}))

aws, spring boot


In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

@tool
def answer_scorer(question: str, answer: str) -> str:
    """Score a student's answer to a placement interview question, 1-10, with one-line rationale.
    Use when evaluating how well a student answered a specific interview question.
    Returns format: 'Score: X/10. Rationale: <reason>'."""
    prompt = (f'Score this placement interview answer 1-10 with one-line rationale.\n'
              f'Question: {question}\n'
              f'Answer: {answer}')
    return llm.invoke(prompt).content

# Test
print(answer_scorer.invoke({
    'question': 'Why TCS Digital?',
    'answer': 'Because TCS is big and they pay well.',
}))

**Score: 2/10**

**Rationale:** The answer is generic, self-serving, and shows no specific research or genuine interest in TCS Digital beyond basic compensation.


In [16]:
from langgraph.prebuilt import create_react_agent

tools = [jd_fetcher, skills_gap, answer_scorer]
agent = create_react_agent(llm, tools=tools)
print(f'Agent created with {len(tools)} tools.')

Agent created with 3 tools.


/tmp/ipykernel_5400/3943891205.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=tools)


In [18]:
profiles = [
    {
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 8.2,
        "skills": ["Python", "Java", "SQL"],
        "target_company": "TCS"
    },
    {
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 8.7,
        "skills": ["Python", "DBMS", "Communication"],
        "target_company": "Cognizant"
    },
    {
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 9.1,
        "skills": ["Python", "AWS", "Machine Learning"],
        "target_company": "Amazon"
    }
]

In [23]:
for i, p in enumerate(profiles):

    print(f'\n{"="*70}')
    print(f'Student {i+1}: {p["name"]}')
    print(f'{"="*70}')

    msg = f"""
    I am {p['name']}, B.Tech {p['branch']}
    CGPA {p['cgpa']}.

    My skills are:
    {', '.join(p['skills'])}

    My target company is:
    {p['target_company']}

    Use the skills_gap tool to identify missing skills
    compared to a strong candidate.

    Use the answer_scorer tool to evaluate this answer:

    Question:
    Why do you want to join the company?

    Answer:
    Because it is a big company with good salary.

    Also generate 3 mock interview questions.
    """

    result = agent.invoke(
        {
            'messages': [('user', msg)]
        },
        config={'recursion_limit': 10}
    )

    for j, m in enumerate(result['messages']):

        print(f'\n[{j}] {type(m).__name__}')

        if hasattr(m, 'content') and m.content:
            print(str(m.content)[:300])

        if hasattr(m, 'tool_calls') and m.tool_calls:
            for tc in m.tool_calls:
                print(
                    f'→ tool_call: '
                    f'{tc.get("name")}({tc.get("args")})'
                )


Student 1: Ravi Kumar

[0] HumanMessage

    I am Ravi Kumar, B.Tech CSE
    CGPA 8.2.

    My skills are:
    Python, Java, SQL

    My target company is:
    TCS

    Use the skills_gap tool to identify missing skills
    compared to a strong candidate.

    Use the answer_scorer tool to evaluate this answer:

    Question:
    Why do y

[1] AIMessage
[{'type': 'text', 'text': "Please provide the 'must-have skills' for a strong candidate at TCS, so I can help you identify the missing skills. In the meantime, I can evaluate your answer and generate mock interview questions.", 'extras': {'signature': 'CuwDAQw51se52iRTj0aXiEPLyIvUlqQmyC1pTLDWshaXJHr

Student 2: Sneha Reddy

[0] HumanMessage

    I am Sneha Reddy, B.Tech ECE
    CGPA 8.7.

    My skills are:
    Python, DBMS, Communication

    My target company is:
    Cognizant

    Use the skills_gap tool to identify missing skills
    compared to a strong candidate.

    Use the answer_scorer tool to evaluate this answer:

    Quest

In [22]:
result = agent.invoke({
    'messages': [('user', 'Fetch this JD and tell me the must-have skills: '
                          'https://this-does-not-exist-99999.example.com/jd')]
}, config={'recursion_limit': 5})

print('Failure recovery trace:')
for j, m in enumerate(result['messages']):
    print(f'\n[{j}] {type(m).__name__}')
    if hasattr(m, 'content') and m.content:
        print(f'    {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        for tc in m.tool_calls:
            print(f'    → {tc.get("name")}({tc.get("args")})')

Failure recovery trace:

[0] HumanMessage
    Fetch this JD and tell me the must-have skills: https://this-does-not-exist-99999.example.com/jd

[1] AIMessage
    → jd_fetcher({'url': 'https://this-does-not-exist-99999.example.com/jd'})

[2] ToolMessage
    ERROR: failed to fetch URL — HTTPSConnectionPool(host='this-does-not-exist-99999.example.com', port=443): Max retries exceeded with url: /jd (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x792ddfd33350>: Failed to resolve 'this-does-not-exist-99999.example.com' ([Errn

[3] AIMessage
    I'm sorry, I was unable to retrieve the job description from the provided URL. It appears the URL is not accessible. Please check the URL and try again.
